# FreshWise: data audit and forecasting baselines
Real weekly orders; inventory and carbon outcomes remain unvalidated. Run cells in order.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root / "src"))
from freshwise.data import load_data, audit, enrich
from freshwise.forecast import backtest, predict
tables = load_data()
audit(tables)

In [ ]:
train = enrich(tables["train.csv"], tables["meal_info.csv"], tables["fulfilment_center_info.csv"])
train.head()

In [ ]:
train.groupby(["center_id", "meal_id"]).week.nunique().describe()

## Ten-week chronological evaluation
Fixed forecast origins; no held-out order outcomes enter the predictions. Last four means last four observed rows. Missing rows are not zeros.

In [ ]:
scores = backtest(train)
scores

In [ ]:
scores.groupby("method")[["mae", "wape", "rmsle"]].mean()

## Emissions: inspect compatible coefficients only
Canada only. Commodity factors do not establish full meal footprints. Recipes and ingredient weights are missing.

In [ ]:
emissions = tables["Emissions intensities.csv"]
intensities = emissions.loc[(emissions["Element"] == "Emissions intensity") & (emissions["Unit"] == "kg CO2eq/kg")]
intensities.pivot(index="Item", columns="Year", values="Value")

In [ ]:
loss = tables["TPFLW.csv"]
loss.groupby("food_supply_stage", dropna=False).size().sort_values(ascending=False)

## Decision gate
Choose forecasting horizon and review baseline errors before adding a model. Inventory, expiry, costs and recipes must be collected or clearly labelled as simulated. Never present simulated savings as observed impact.